# 🚦 Assignment 12 — Pakistani Traffic Object Detection & Tracking (YOLO)

This notebook tests the custom-trained YOLO model (`Pakistani_Trafic_V2.pt`) on Pakistani road traffic:
1. Load the model and inspect its classes
2. Run detection on an image
3. Run multi-object **tracking** on a video and count **unique** vehicles per class
4. Compare the two bundled checkpoints

> All paths are relative to this folder, so the notebook runs on any machine.

In [ ]:
# !pip install -r requirements.txt   # uncomment on a fresh environment

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # avoids an OpenMP clash on Windows/Anaconda

from pathlib import Path
from collections import defaultdict
import time

import cv2
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

PROJECT_DIR = Path.cwd()
MODEL_V2 = PROJECT_DIR / "Pakistani_Trafic_V2.pt"
MODEL_V1 = PROJECT_DIR / "Pakistan_Traffic_Model.pt"

# Put your own test media here (not included in the repo)
TEST_VIDEO = PROJECT_DIR / "tf_test_1.mp4"
TEST_IMAGE = PROJECT_DIR / "test_image.jpg"

## 1. Load the model and inspect the classes

In [ ]:
model = YOLO(str(MODEL_V2))
print("Task:", model.task)
pd.DataFrame(list(model.names.items()), columns=["class_id", "class_name"])

## 2. Detection on a single image

In [ ]:
def show(bgr, title=""):
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

if TEST_IMAGE.exists():
    res = model.predict(str(TEST_IMAGE), conf=0.25, iou=0.45, verbose=False)[0]
    counts = pd.Series([model.names[int(c)] for c in res.boxes.cls]).value_counts()
    show(res.plot(), f"{len(res.boxes)} objects detected")
    display(counts.rename("count").to_frame())
else:
    print(f"Add an image at {TEST_IMAGE.name} to run this cell.")

## 3. Tracking on a video + unique vehicle counting

`stream=True` yields one result per frame instead of keeping every frame in RAM
(the original `model.track(..., save=True)` warned about this).
Counting **unique track IDs** avoids counting the same car once per frame.

In [ ]:
MAX_FRAMES = 300   # set to None for the full video

if TEST_VIDEO.exists():
    unique_ids = defaultdict(set)
    per_frame = []
    t0 = time.time()
    for i, r in enumerate(model.track(str(TEST_VIDEO), tracker="bytetrack.yaml",
                                      conf=0.25, iou=0.45, stream=True, verbose=False)):
        if MAX_FRAMES and i >= MAX_FRAMES:
            break
        if r.boxes.id is not None:
            for cls, tid in zip(r.boxes.cls.int().tolist(), r.boxes.id.int().tolist()):
                unique_ids[model.names[cls]].add(tid)
        per_frame.append(len(r.boxes))
        last_frame = r
    elapsed = time.time() - t0
    print(f"Processed {len(per_frame)} frames in {elapsed:.1f}s ({len(per_frame)/elapsed:.1f} FPS)")

    summary = (pd.DataFrame({"class": list(unique_ids), "unique_objects": [len(v) for v in unique_ids.values()]})
                 .sort_values("unique_objects", ascending=False).reset_index(drop=True))
    display(summary)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(per_frame); ax[0].set(title="Detections per frame", xlabel="frame", ylabel="objects")
    ax[1].bar(summary["class"], summary["unique_objects"]); ax[1].set(title="Unique tracked objects per class")
    plt.tight_layout(); plt.show()
    show(last_frame.plot(), "Last processed frame")
else:
    print(f"Add a video at {TEST_VIDEO.name} to run this cell.")

## 4. Compare the two checkpoints (speed & detections on the same image/frame)

In [ ]:
sample = TEST_IMAGE if TEST_IMAGE.exists() else None
if sample is None and TEST_VIDEO.exists():
    ok, frame = cv2.VideoCapture(str(TEST_VIDEO)).read()
    sample = frame if ok else None

if sample is not None:
    rows = []
    for path in (MODEL_V1, MODEL_V2):
        m = YOLO(str(path))
        m.predict(sample, verbose=False)            # warm-up
        t0 = time.time(); r = m.predict(sample, conf=0.25, verbose=False)[0]
        rows.append({"model": path.name, "size_MB": round(path.stat().st_size / 1e6, 1),
                     "objects": len(r.boxes), "latency_ms": round((time.time() - t0) * 1000, 1)})
    display(pd.DataFrame(rows))
else:
    print("Add a test image or video to compare the checkpoints.")

## Conclusion
- The V2 checkpoint is larger than V1, which usually means a bigger backbone: more accurate but slower.
- Tracking plus unique-ID counting gives realistic traffic counts; per-frame counts overcount heavily.
- The full interactive app (image / video / webcam / reports) is in `app.py` → `python app.py`.